# SwinUNETR 3D Inference
Semantic segmentation → 3D connected components → instance masks

In [6]:
import os
import re
import numpy as np
import tifffile
import torch
from pathlib import Path
from scipy.ndimage import label as ndimage_label, generate_binary_structure
from skimage.morphology import remove_small_objects
from monai.inferers import sliding_window_inference
from monai.networks.nets import SwinUNETR

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — change these two lines only
# ─────────────────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = "/work/scratch/rawat/models/monai_swinunetr/swinunetr_calyx_Original/swinunetr_calyx_Original_best.pth"
INPUT_FOLDER    = "/home/students/rawat/Final_dataset/Original/test/images"
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_BASE            = "/home/students/rawat/Results"
VALID_XY_SIZES         = {1100, 1101}
BLOB_REMOVAL_THRESHOLD = 1450

# Must match training CONFIG exactly
PATCH_SIZE    = (32, 192, 192)
SW_BATCH_SIZE = 2
IN_CHANNELS   = 2
OUT_CHANNELS  = 2
FEATURE_SIZE  = 48

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract model name from path: .../monai_swinunetr/<MODEL_NAME>/...
_match = re.search(r'/monai_swinunetr/([^/]+)/', CHECKPOINT_PATH)
MODEL_NAME = _match.group(1) if _match else Path(CHECKPOINT_PATH).parent.name

print(f"Device       : {DEVICE}")
print(f"Model name   : {MODEL_NAME}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Input folder : {INPUT_FOLDER}")
print(f"Output base  : {OUTPUT_BASE}/{MODEL_NAME}/")

Device       : cuda
Model name   : swinunetr_calyx_Original
Checkpoint   : /work/scratch/rawat/models/monai_swinunetr/swinunetr_calyx_Original/swinunetr_calyx_Original_best.pth
Input folder : /home/students/rawat/Final_dataset/Original/test/images
Output base  : /home/students/rawat/Results/swinunetr_calyx_Original/


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# PREPROCESSING HELPERS — match training pipeline exactly
# ─────────────────────────────────────────────────────────────────────────────

def to_zcyx(arr: np.ndarray) -> np.ndarray:
    """Normalise any TIFF to (Z, C, Y, X)."""
    if arr.ndim == 3:
        return arr[:, np.newaxis]
    if arr.shape[0] <= 4 and arr.shape[0] < arr.shape[1]:
        arr = arr.transpose(1, 0, 2, 3)     # (C,Z,Y,X) → (Z,C,Y,X)
    return arr

def percentile_normalize(img: np.ndarray) -> np.ndarray:
    """Per-channel normalization over nonzero voxels — matches PercentileNormalized in training.
    Applied even if input is already [0,1]: consistent with training which did the same."""
    out = np.zeros_like(img, dtype=np.float32)
    for c in range(img.shape[0]):
        ch      = img[c]
        nonzero = ch[ch > 0]
        if nonzero.size == 0:
            out[c] = ch
            continue
        lo = float(np.percentile(nonzero, 0.00001))
        hi = float(np.percentile(nonzero, 99.99999))
        out[c] = np.clip((ch - lo) / (hi - lo), 0.0, 1.0) if hi > lo else ch
    return out

def preprocess(path: Path) -> torch.Tensor:
    """Load → axis fix → (C,Z,Y,X) → normalize → tensor (1,C,Z,Y,X)."""
    arr = tifffile.imread(str(path))
    arr = to_zcyx(arr)                          # → (Z,C,Y,X)
    arr = arr.transpose(1, 0, 2, 3)             # → (C,Z,Y,X) for MONAI
    arr = arr.astype(np.float32)
    arr = percentile_normalize(arr)
    return torch.from_numpy(arr).unsqueeze(0)   # → (1,C,Z,Y,X)

print("Helper functions ready.")

Helper functions ready.


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# SCAN FOLDER FOR VALID IMAGES
# ─────────────────────────────────────────────────────────────────────────────
all_tifs = sorted(Path(INPUT_FOLDER).glob("*.tif"))
valid_images = []

print(f"Scanning: {INPUT_FOLDER}\n")
for p in all_tifs:
    arr = tifffile.imread(str(p))
    y, x = arr.shape[-2], arr.shape[-1]
    if y in VALID_XY_SIZES and x in VALID_XY_SIZES:
        valid_images.append(p)
        print(f"  VALID   {p.name}  {arr.shape}")
    else:
        print(f"  SKIP    {p.name}  {arr.shape}  (XY={y}x{x})")

print(f"\n{len(valid_images)} image(s) will be processed.")

Scanning: /home/students/rawat/Final_dataset/Original/test/images

  SKIP    Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL.tif  (70, 2, 1834, 1834)  (XY=1834x1834)
  VALID   Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100.tif  (70, 2, 1100, 1100)
  VALID   PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.tif  (2, 76, 1101, 1101)

2 image(s) will be processed.


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD MODEL ONCE
# ─────────────────────────────────────────────────────────────────────────────
print(f"Loading model: {MODEL_NAME} ...")
model = SwinUNETR(
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    feature_size=FEATURE_SIZE,
    use_checkpoint=False,
).to(DEVICE)

state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
# _best.pth = state_dict only; full checkpoints have 'model_state_dict' key
if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]
model.load_state_dict(state)
model.eval()
print("Model ready.")

Loading model: swinunetr_calyx_Original ...
Model ready.


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE LOOP — all valid images
# ─────────────────────────────────────────────────────────────────────────────
for img_path in valid_images:
    print(f"\n{'─'*60}")
    print(f"Processing: {img_path.name}")

    # ── Preprocess ───────────────────────────────────────────────────────────
    tensor = preprocess(img_path).to(DEVICE)    # (1,C,Z,Y,X)
    print(f"  Input tensor : {tuple(tensor.shape)}")

    # ── Sliding window inference ──────────────────────────────────────────────
    with torch.no_grad():
        output = sliding_window_inference(
            tensor,
            roi_size=PATCH_SIZE,
            sw_batch_size=SW_BATCH_SIZE,
            predictor=model,
            overlap=0.5,
        )

    # ── Semantic → binary foreground mask ────────────────────────────────────
    # output: (1, OUT_CHANNELS, Z, Y, X)
    # argmax over channel dim → class index (0=bg, 1=fg) → (Z,Y,X) bool
    binary = output[0].argmax(dim=0).cpu().numpy() == 1

    # ── 3D connected components → instance segmentation ──────────────────────
    structure = generate_binary_structure(3, 3)  # 26-connectivity
    labeled, n_obj = ndimage_label(binary, structure=structure)
    print(f"  After 3D CC        : {n_obj} objects")

    # ── Blob removal ──────────────────────────────────────────────────────────
    clean = remove_small_objects(labeled.astype(bool), min_size=BLOB_REMOVAL_THRESHOLD, connectivity=1)
    clean, n_final = ndimage_label(clean, structure=structure)
    print(f"  After blob removal : {n_final} objects  (threshold={BLOB_REMOVAL_THRESHOLD} vox)")

    # ── Save ──────────────────────────────────────────────────────────────────
    out_path = Path(OUTPUT_BASE) / MODEL_NAME / f"{img_path.stem}_mask.tif"
    os.makedirs(out_path.parent, exist_ok=True)
    dtype = np.uint32 if n_final > 65535 else np.uint16
    tifffile.imwrite(str(out_path), clean.astype(dtype))
    print(f"  Saved → {out_path}")

print(f"\n{'='*60}")
print(f"Done. Processed {len(valid_images)} image(s).")


────────────────────────────────────────────────────────────
Processing: Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100.tif
  Input tensor : (1, 2, 70, 1100, 1100)
  After 3D CC        : 1289 objects
  After blob removal : 16 objects  (threshold=1450 vox)
  Saved → /home/students/rawat/Results/swinunetr_calyx_Original/Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100_mask.tif

────────────────────────────────────────────────────────────
Processing: PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.tif
  Input tensor : (1, 2, 76, 1101, 1101)
  After 3D CC        : 1862 objects
  After blob removal : 22 objects  (threshold=1450 vox)
  Saved → /home/students/rawat/Results/swinunetr_calyx_Original/PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit_mask.tif

Done. Processed 2 image(s).
